# Basic machine learning: from examples to a trained model

This short lab prepares you for the transformer notebooks. You will:

- turn a physics-inspired problem into inputs and targets;
- compare a linear classifier with a small neural network;
- train with cross-entropy and gradient-based updates;
- measure generalization on data the model did not see.

The dataset below is an **analytic teaching fixture**, not a scientific simulation.

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 2603
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

plt.rcParams.update({
    "figure.figsize": (7.4, 4.6),
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = np.array(["#2D617F", "#B58A35", "#B95535"])
DEVICE = torch.device("cpu")

## 1. A nonlinear classification problem

For a point $(q,p)$, define the dimensionless teaching energy

$$
E(q,p) = \frac{q^2+p^2}{2}.
$$

We label three energy regimes:

$$
y =
\begin{cases}
0, & E < 0.5,\\
1, & 0.5 \le E < 1.5,\\
2, & E \ge 1.5.
\end{cases}
$$

The correct boundaries are circles. This gives us a controlled question:
can a model learn a nonlinear boundary from labelled examples?

In [ ]:
N = 3000
x_np = rng.normal(size=(N, 2)).astype(np.float32)
energy = 0.5 * np.sum(x_np**2, axis=1)
y_np = np.digitize(energy, bins=[0.5, 1.5]).astype(np.int64)

fig, ax = plt.subplots()
for label, name in enumerate(["low", "middle", "high"]):
    mask = y_np == label
    ax.scatter(
        x_np[mask, 0], x_np[mask, 1],
        s=12, alpha=0.55, color=COLORS[label], label=name,
    )
ax.set(xlabel="$q$", ylabel="$p$", aspect="equal",
       title="Examples and their target energy regime")
ax.legend(frameon=False, ncols=3, loc="upper center")
plt.show()

print("input shape:", x_np.shape)
print("target shape:", y_np.shape)
print("class fractions:", np.bincount(y_np) / N)

## 2. Split first, then standardize

- **Training set:** fit the parameters.
- **Validation set:** compare modelling choices.
- **Test set:** report the final result once.

Standardization uses only the training mean and standard deviation. Using
validation or test statistics would leak information into training.

In [ ]:
indices = rng.permutation(N)
n_train, n_val = 1800, 600
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

train_mean = x_np[train_idx].mean(axis=0, keepdims=True)
train_std = x_np[train_idx].std(axis=0, keepdims=True)
x_scaled = (x_np - train_mean) / train_std

def as_tensors(idx):
    return (
        torch.tensor(x_scaled[idx], dtype=torch.float32, device=DEVICE),
        torch.tensor(y_np[idx], dtype=torch.long, device=DEVICE),
    )

x_train, y_train = as_tensors(train_idx)
x_val, y_val = as_tensors(val_idx)
x_test, y_test = as_tensors(test_idx)

assert torch.allclose(x_train.mean(0), torch.zeros(2), atol=2e-6)
assert torch.allclose(x_train.std(0, unbiased=False), torch.ones(2), atol=2e-6)

majority_baseline = torch.bincount(y_train).max().item() / len(y_train)
print("train / validation / test:", len(y_train), len(y_val), len(y_test))
print(f"majority-class baseline: {majority_baseline:.3f}")

## 3. The model returns logits

A classifier maps each input to three real numbers called **logits**:

$$
(q,p) \longrightarrow (z_0,z_1,z_2).
$$

The largest logit gives the predicted class. `CrossEntropyLoss` compares the
logits with the correct integer target and supplies one scalar loss.

A linear classifier can draw straight boundaries only. We will use it as a
baseline before adding nonlinear hidden layers.

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.output = nn.Linear(2, 3)

    def forward(self, x):
        return self.output(x)



class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO 1: build a small network with
        # 2 inputs -> 32 hidden units -> 32 hidden units -> 3 logits.
        # Put nn.Tanh() after each hidden linear layer.
        self.net = nn.Sequential(
            # nn.Linear(...),
            # nn.Tanh(),
            # ...
        )

    def forward(self, x):
        return self.net(x)


linear_model = LinearClassifier().to(DEVICE)
mlp_model = SmallMLP().to(DEVICE)

sample_logits = linear_model(x_train[:5])
print("five inputs:", x_train[:5].shape)
print("five sets of logits:", sample_logits.shape)

mlp_sample_logits = mlp_model(x_train[:5])
assert mlp_sample_logits.shape == (5, 3), (
    "The completed MLP should return three logits for every input."
)

## 4. Training is predict, compare, update

For a minibatch:

1. make predictions;
2. compare them with the targets;
3. compute gradients by backpropagation;
4. let the optimizer update the parameters.

The validation set is evaluated without gradient tracking and never updates the model.
The function restores the checkpoint with the smallest validation loss; the
curves still show every epoch.

In [ ]:
def evaluate_loss(model, x, y, loss_fn):
    model.eval()
    with torch.no_grad():
        return loss_fn(model(x), y).item()


def train_model(model, *, epochs=120, learning_rate=1e-2):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=128,
        shuffle=True,
        generator=generator,
    )

    history = {"train": [], "val": []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

        # TODO 2: clear old gradients, compute new gradients,
        # and ask the optimizer to update the parameters.
        # optimizer.____()
        # loss.____()
        # optimizer.____()
        # Remove the NotImplementedError after filling the three lines.
        raise NotImplementedError("Complete the three update steps.")

        train_loss = evaluate_loss(model, x_train, y_train, loss_fn)
        val_loss = evaluate_loss(model, x_val, y_val, loss_fn)
        history["train"].append(train_loss)
        history["val"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history



def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        predicted_class = model(x).argmax(dim=1)
    # TODO 3: return the fraction of correctly classified examples.
    # return ...
    # Remove the NotImplementedError after adding the return statement.
    raise NotImplementedError("Compute the classification accuracy.")

## 5. Compare the linear baseline with the nonlinear model

Use the same split, loss, optimizer, and number of epochs. The controlled
difference is the model family.

In [ ]:
linear_history = train_model(linear_model)
mlp_history = train_model(mlp_model)

linear_val_acc = accuracy(linear_model, x_val, y_val)
mlp_val_acc = accuracy(mlp_model, x_val, y_val)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), sharey=True)
for ax, history, title in zip(
    axes,
    [linear_history, mlp_history],
    ["Linear classifier", "Small MLP"],
):
    ax.plot(history["train"], label="training", color="#2D617F", lw=2)
    ax.plot(history["val"], label="validation", color="#B95535", lw=2)
    ax.set(xlabel="epoch", ylabel="cross-entropy loss", title=title)
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"linear validation accuracy: {linear_val_acc:.3f}")
print(f"MLP validation accuracy:    {mlp_val_acc:.3f}")

assert mlp_val_acc > 0.90, "The MLP should learn the three nonlinear regions."
assert mlp_val_acc > linear_val_acc + 0.35, (
    "The nonlinear model should clearly improve on the linear baseline."
)

## 6. Inspect the learned decision boundaries

The linear model is deliberately limited. The MLP can bend its boundaries
because its hidden layers include nonlinear activations.

In [ ]:
def decision_map(model, *, limit=3.2, steps=240):
    q = np.linspace(-limit, limit, steps)
    p = np.linspace(-limit, limit, steps)
    qq, pp = np.meshgrid(q, p)
    grid_original = np.column_stack([qq.ravel(), pp.ravel()]).astype(np.float32)
    grid_scaled = (grid_original - train_mean) / train_std
    with torch.no_grad():
        predicted = model(
            torch.tensor(grid_scaled, dtype=torch.float32)
        ).argmax(1).numpy()
    return qq, pp, predicted.reshape(qq.shape)


fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.2), constrained_layout=True)
xx = np.linspace(-3.2, 3.2, 240)
yy = np.linspace(-3.2, 3.2, 240)
qq, pp = np.meshgrid(xx, yy)
truth = np.digitize(0.5 * (qq**2 + pp**2), bins=[0.5, 1.5])

panels = [
    (truth, "analytic target"),
    (decision_map(linear_model)[2], "linear classifier"),
    (decision_map(mlp_model)[2], "small MLP"),
]
for ax, (region, title) in zip(axes, panels):
    ax.contourf(qq, pp, region, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=COLORS, alpha=0.34)
    ax.scatter(x_val[:220, 0] * train_std[0, 0] + train_mean[0, 0],
               x_val[:220, 1] * train_std[0, 1] + train_mean[0, 1],
               c=COLORS[y_val[:220].numpy()], s=8, alpha=0.65)
    ax.set(xlabel="$q$", ylabel="$p$", title=title, aspect="equal",
           xlim=(-3.2, 3.2), ylim=(-3.2, 3.2))
plt.show()

## 7. Logits become probabilities for interpretation

For one input, softmax converts the logits into non-negative probabilities
that sum to one:

$$
p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}.
$$

PyTorch's cross-entropy loss expects raw logits, so do **not** apply softmax
before the loss. We use softmax only when inspecting predictions. These are
the model's normalized class probabilities, not automatically calibrated
uncertainty estimates.

In [ ]:
probe_points = np.array([
    [0.2, 0.1],   # low energy
    [1.0, 0.5],   # middle energy
    [2.1, 0.2],   # high energy
], dtype=np.float32)
probe_scaled = (probe_points - train_mean) / train_std

mlp_model.eval()
with torch.no_grad():
    probe_logits = mlp_model(torch.tensor(probe_scaled))
    probe_probabilities = torch.softmax(probe_logits, dim=1)

for point, logits, probabilities in zip(
    probe_points, probe_logits, probe_probabilities
):
    print(f"(q, p) = {tuple(point)}")
    print("  logits:       ", np.round(logits.numpy(), 3))
    print("  probabilities:", np.round(probabilities.numpy(), 3))

test_accuracy = accuracy(mlp_model, x_test, y_test)
print(f"\nfinal test accuracy: {test_accuracy:.3f}")

## 8. What carries over to transformers?

For the token transformer used in the next notebooks, the path becomes:

$$
\text{token IDs} \rightarrow \text{embeddings} \rightarrow \text{attention}
\rightarrow \text{MLP} \rightarrow \text{logits}
\rightarrow \text{loss} \rightarrow \text{gradients} \rightarrow \text{update}.
$$

The input representation and model internals change, but the
loss–gradient–update loop is the same.

Before moving on, answer briefly:

1. Why did the MLP outperform the linear classifier?
2. Why must the test set remain untouched until the end?
3. Which object has shape `(batch, 3)`, and what do its three entries mean?

References: [PyTorch autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html),
[PyTorch `CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html),
and [*Deep Learning*](https://www.deeplearningbook.org/).